# FB Group Monitoring Platform — Colab Runner

Jalankan setiap cell **satu per satu** secara berurutan.

| # | Cell | Keterangan |
|---|------|------------|
| 1 | Clone & Setup | Clone repo + install dependencies |
| 2 | `logger_setup.py` | Setup logging |
| 3 | `config_loader.py` | Load konfigurasi settings.yaml |
| 4 | `database.py` | Init database SQLite |
| 5 | `session.py` | Cek status sesi login |
| 6 | `auth.py` | Login Facebook (buat/cek sesi) |
| 7 | `filter_engine.py` | Test filter keyword engine |
| 8 | `processor.py` | Test post processor |
| 9 | `notifier.py` | Test notifier lokal |
| 10 | `telegram_notifier.py` | Test Telegram notifier |
| 11 | `collector.py` | Test fetch feed grup |
| 12 | `list_groups.py` | List grup yang diikuti |
| 13 | `watcher.py` | Test 1 siklus watcher |
| 14 | `main.py` | Jalankan monitoring penuh |

---
## Cell 1 — Clone Repo & Install Dependencies

In [ ]:
# Clone repo
!git clone https://github.com/Herutriana44/fb_group_monitoring_platform
%cd fb_group_monitoring_platform

# Install Python dependencies
!pip install -q -r requirements.txt

# Install Playwright + Chromium browser + system deps
!playwright install chromium
!playwright install --with-deps chromium

print("\n✓ Setup selesai!")

---
## Cell 2 — `logger_setup.py`
Setup logging terpusat (console + rotating file). Jalankan ini sekali sebelum modul lain.

In [ ]:
import sys, os
sys.path.insert(0, 'src')
os.chdir('/content/fb_group_monitoring_platform')  # pastikan working dir benar

import logging
from src.logger_setup import setup_logging, get_logger

setup_logging(level=logging.INFO)
log = get_logger("colab")

log.debug("Pesan DEBUG")
log.info("Pesan INFO")
log.warning("Pesan WARNING")
log.error("Pesan ERROR")

print("\n✓ logger_setup.py OK")

---
## Cell 3 — `config_loader.py`
Load dan tampilkan konfigurasi dari `config/settings.yaml`.

In [ ]:
import sys
sys.path.insert(0, 'src')

from src.config_loader import load_config

cfg = load_config()

print("=== Config Loaded ===")
print(f"  DB Path         : {cfg.database.path}")
print(f"  Browser Headless: {cfg.browser.headless}")
print(f"  Browser Timeout : {cfg.browser.timeout}ms")
print(f"  Check Interval  : {cfg.monitoring.interval_seconds}s")
print(f"  Group IDs       : {cfg.monitoring.group_ids}")
print(f"  Keywords        : {cfg.monitoring.keywords}")
print(f"  Telegram Token  : {'SET' if cfg.telegram.bot_token else 'NOT SET'}")
print(f"  Telegram Chat ID: {'SET' if cfg.telegram.chat_id else 'NOT SET'}")
print(f"  Webhook URL     : {'SET' if cfg.webhook.url else 'NOT SET'}")
print("\n✓ config_loader.py OK")

---
## Cell 4 — `database.py`
Inisialisasi database SQLite dan jalankan test CRUD (posts, keywords, logs, sessions).

In [ ]:
import sys, asyncio
sys.path.insert(0, 'src')

from src.database import Database

async def test_database():
    db = Database("data/monitor.db")
    await db.init()

    # Keywords
    print("=== Keywords ===")
    await db.add_keyword("lelang")
    await db.add_keyword("jual")
    await db.add_keyword("lelang")  # duplikat
    keywords = await db.get_active_keywords()
    print(f"  Active keywords: {keywords}")

    # Posts
    print("\n=== Posts ===")
    saved = await db.save_post("post_001", "group_123", "Lelang sepatu murah!", "UserA", "http://fb.com/1")
    print(f"  Post saved    : {saved}")
    dup   = await db.save_post("post_001", "group_123", "Duplikat", "UserA")
    print(f"  Dup blocked   : {not dup}")
    exists = await db.post_exists("post_001")
    print(f"  post_exists   : {exists}")
    await db.mark_notified("post_001")
    posts = await db.get_recent_posts()
    print(f"  Recent posts  : {len(posts)}, notified={posts[0]['notified']}")

    # Logs
    print("\n=== Logs ===")
    await db.write_log("INFO", "Monitoring started")
    await db.write_log("ERROR", "Connection timeout")
    logs = await db.get_logs(limit=5)
    print(f"  Logs saved    : {len(logs)}")

    # Stats
    print("\n=== Stats ===")
    stats = await db.get_stats()
    for k, v in stats.items():
        print(f"  {k}: {v}")

    print("\n✓ database.py OK")

await test_database()

---
## Cell 5 — `session.py`
Cek status file sesi yang tersimpan (apakah ada, dan apakah masih valid).

In [ ]:
import sys, asyncio, os
sys.path.insert(0, 'src')

from src.session import SessionManager
from playwright.async_api import async_playwright

async def test_session():
    sm = SessionManager()
    session_file = sm.session_file

    print(f"Session file path: {session_file}")

    if not os.path.exists(session_file):
        print("⚠  File sesi tidak ditemukan.")
        print("   Jalankan Cell 6 (auth.py) untuk login dan buat sesi.")
        return

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context()

        loaded = await sm.load(context)
        print(f"Load session : {loaded}")

        if loaded:
            print("Mengecek validitas sesi (membuka Facebook)...")
            valid = await sm.is_valid(context)
            print(f"Session valid: {valid}")
            if valid:
                print("✓ Sesi masih aktif — siap digunakan.")
            else:
                print("✗ Sesi expired — jalankan Cell 6 untuk login ulang.")

        await browser.close()

await test_session()

---
## Cell 6 — `auth.py`
Login ke Facebook dan simpan sesi ke `data/session.json`.

> ⚠️ **Isi `FB_EMAIL` dan `FB_PASSWORD` dengan akun Facebook kamu sebelum menjalankan cell ini.**

In [ ]:
import sys, asyncio
sys.path.insert(0, 'src')

# ── ATUR KREDENSIAL DI SINI ──────────────────────────────────────────────────
FB_EMAIL    = "email_kamu@example.com"   # ganti dengan email Facebook
FB_PASSWORD = "password_kamu"            # ganti dengan password Facebook
# ─────────────────────────────────────────────────────────────────────────────

from src.auth import do_login, do_check

async def run_auth():
    print(f"Login sebagai: {FB_EMAIL}")
    success = await do_login(FB_EMAIL, FB_PASSWORD, headless=True)
    if success:
        print("✓ Login berhasil! Sesi disimpan ke data/session.json")
    else:
        print("✗ Login gagal. Cek email/password atau tangani 2FA.")

    print("\nVerifikasi sesi...")
    await do_check()

await run_auth()

---
## Cell 7 — `filter_engine.py`
Test FilterEngine dan AutoCommentEngine dengan data dummy.

In [ ]:
import sys
sys.path.insert(0, 'src')

from src.filter_engine import FilterEngine, AutoCommentEngine, PostData

# Test FilterEngine
print("=== FilterEngine Test ===\n")

engine = FilterEngine(
    keywords=["lelang", "jual", "auction"],
    blacklist=["spam", "promosi berbayar"],
    min_length=10,
)

test_posts = [
    PostData("p001", "grp1", "Lelang sepatu Nike size 42, start 50rb!",  "UserA", "http://fb.com/p001"),
    PostData("p002", "grp1", "Jual motor bekas harga nego",               "UserB", "http://fb.com/p002"),
    PostData("p003", "grp1", "Selamat pagi semua!",                       "UserC", "http://fb.com/p003"),
    PostData("p004", "grp1", "ini spam promosi berbayar lelang palsu",    "UserD", "http://fb.com/p004"),
    PostData("p005", "grp1", "Hi",                                         "UserE", "http://fb.com/p005"),
    PostData("p006", "grp1", "Auction jam tangan mewah mulai 100rb",      "UserF", "http://fb.com/p006"),
]

all_results = [engine.evaluate(p) for p in test_posts]
print("Semua evaluasi:")
for r in all_results:
    print(f"  {r}")

matched = engine.filter_posts(test_posts)
print(f"\nTotal match: {len(matched)}")
for r in matched:
    print(f"  ✓ {r.post.content[:50]} | keywords: {r.matched_keywords}")

# Test AutoCommentEngine
print("\n=== AutoCommentEngine Test ===")
comment_engine = AutoCommentEngine(comment_template="Saya minat! Post: {post_id} oleh {author}")
comment = comment_engine._build_comment(test_posts[0])
print(f"  Template result: {comment}")

print("\n✓ filter_engine.py OK")

---
## Cell 8 — `processor.py`
Test PostProcessor dengan data dummy.

In [ ]:
import sys, logging
sys.path.insert(0, 'src')
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")

from src.processor import PostProcessor

processor = PostProcessor(keywords=["lelang", "jual", "auction"])

dummy_posts = [
    "Lelang sepatu Nike Air Max, start 150rb!",
    "Selamat pagi semua, semoga harimu menyenangkan.",
    "JUAL MURAH laptop second core i5, RAM 8GB, SSD 256GB.",
    "Cari teman nongkrong di Bandung.",
    "Auction hari ini jam 3 sore! Barang langka!",
]

matches = processor.process_posts(dummy_posts)

print(f"Total post    : {len(dummy_posts)}")
print(f"Total match   : {len(matches)}")
print("\nPost yang match:")
for m in matches:
    print(f"  - {m[:80]}")

print("\n✓ processor.py OK")

---
## Cell 9 — `notifier.py`
Test notifier lokal (output ke console).

In [ ]:
import sys, logging
sys.path.insert(0, 'src')
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")

from src.notifier import Notifier

notifier = Notifier()

test_messages = [
    "Match found: Lelang sepatu Nike Air Max, start 150rb!",
    "Match found: JUAL MURAH laptop second core i5, RAM 8GB.",
    "System: Monitoring cycle selesai. 2 post baru ditemukan.",
]

for msg in test_messages:
    notifier.notify(msg)

print("✓ notifier.py OK")

---
## Cell 10 — `telegram_notifier.py`
Test koneksi Telegram dan kirim pesan notifikasi.

> ⚠️ **Isi `BOT_TOKEN` dan `CHAT_ID` dari `config/settings.yaml` atau @BotFather sebelum menjalankan.**

In [ ]:
import sys, asyncio
sys.path.insert(0, 'src')

# ── ATUR TELEGRAM CONFIG DI SINI ─────────────────────────────────────────────
BOT_TOKEN = ""   # contoh: "123456789:ABCDefgh..."
CHAT_ID   = ""   # contoh: "987654321"
# ─────────────────────────────────────────────────────────────────────────────

from src.telegram_notifier import TelegramNotifier

async def test_telegram():
    if not BOT_TOKEN or not CHAT_ID:
        print("⚠  BOT_TOKEN dan CHAT_ID belum diisi. Skip test Telegram.")
        print("   Isi variabel BOT_TOKEN dan CHAT_ID di atas lalu jalankan ulang.")
        return

    notifier = TelegramNotifier(bot_token=BOT_TOKEN, chat_id=CHAT_ID)

    print("Testing koneksi bot...")
    ok = await notifier.test_connection()
    print(f"  Koneksi      : {'✓ OK' if ok else '✗ GAGAL'}")

    if ok:
        print("Mengirim pesan test...")
        sent = await notifier.send_message("✅ <b>Test notifikasi FB Monitor berhasil!</b>")
        print(f"  Kirim pesan  : {'✓ OK' if sent else '✗ GAGAL'}")

        print("Mengirim post alert test...")
        sent2 = await notifier.send_post_alert(
            post_content="Lelang sepatu Nike Air Max 42, kondisi 9/10, start 200rb no reserve!",
            post_url="https://facebook.com/groups/contoh/posts/123",
            group_id="lelang_barang_bekas",
            matched_keywords=["lelang", "nike"],
            author="Budi Santoso",
        )
        print(f"  Post alert   : {'✓ OK' if sent2 else '✗ GAGAL'}")

    print("\n✓ telegram_notifier.py OK")

await test_telegram()

---
## Cell 11 — `collector.py`
Fetch feed dari grup Facebook menggunakan sesi yang sudah login.

> ⚠️ **Pastikan Cell 6 (auth.py) sudah dijalankan dan sesi valid. Isi `GROUP_ID` di bawah.**

In [ ]:
import sys, asyncio, json, os
sys.path.insert(0, 'src')

# ── ATUR GROUP ID DI SINI ────────────────────────────────────────────────────
GROUP_ID = "27422186357398808"   # ganti dengan group ID target
# ─────────────────────────────────────────────────────────────────────────────

from src.collector import GroupCollector
from src.session import SessionManager
from playwright.async_api import async_playwright

async def test_collector():
    sm = SessionManager()
    session_exists = os.path.exists(sm.session_file)
    print(f"Session file  : {sm.session_file}")
    print(f"Session ada   : {session_exists}")

    collector = GroupCollector(group_id=GROUP_ID)
    print(f"\nFetching feed dari group: {GROUP_ID}")
    print(f"URL: {collector.base_url}")

    posts = await collector.fetch_feed()

    print(f"\nTotal post ditemukan: {len(posts)}")
    if posts:
        print("\nPratinjau 3 post pertama:")
        for i, post in enumerate(posts[:3], 1):
            preview = post[:150].replace('\n', ' ')
            print(f"  [{i}] {preview}...")
    else:
        print("  (Tidak ada post — pastikan sesi valid dan group ID benar)")

    print("\n✓ collector.py OK")

await test_collector()

---
## Cell 12 — `list_groups.py`
Tampilkan daftar grup Facebook yang diikuti oleh akun yang sudah login.

> ⚠️ **Pastikan Cell 6 (auth.py) sudah dijalankan terlebih dahulu.**

In [ ]:
import sys, asyncio
sys.path.insert(0, 'src')

from src.list_groups import list_groups

print("Mengambil daftar grup yang diikuti...\n")
groups = await list_groups()

if groups:
    print(f"\nTotal grup ditemukan: {len(groups)}")
    print("\nDaftar grup:")
    for url, name in groups.items():
        print(f"  - {name}: {url}")
else:
    print("Tidak ada grup ditemukan atau sesi tidak valid.")

print("\n✓ list_groups.py OK")

---
## Cell 13 — `watcher.py`
Test 1 siklus fetch watcher tanpa filter/notifier.

> ⚠️ **Isi `GROUP_ID` dengan grup yang ingin dipantau.**

In [ ]:
import sys, asyncio, os
sys.path.insert(0, 'src')

# ── ATUR GROUP ID DI SINI ────────────────────────────────────────────────────
GROUP_ID = "27422186357398808"   # ganti dengan group ID target
# ─────────────────────────────────────────────────────────────────────────────

from src.watcher import GroupWatcher
from src.session import SessionManager
from playwright.async_api import async_playwright

async def test_watcher():
    session_file = os.path.join('data', 'session.json')

    if not os.path.exists(session_file):
        print("⚠  Sesi tidak ditemukan. Jalankan Cell 6 (auth.py) dulu.")
        return

    watcher = GroupWatcher(
        group_ids=[GROUP_ID],
        session_file=session_file,
        headless=True,
    )

    print(f"Testing 1 siklus fetch untuk group: {GROUP_ID}")

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context()

        sm = SessionManager(session_file=session_file)
        await sm.load(context)

        page = await context.new_page()
        posts = await watcher._fetch_group(page, GROUP_ID)

        await browser.close()

    print(f"\n=== Hasil Fetch ===")
    print(f"Total post ditemukan: {len(posts)}")
    for i, post in enumerate(posts[:3], 1):
        print(f"\n[Post {i}]")
        print(f"  ID     : {post['post_id']}")
        print(f"  Author : {post['author']}")
        print(f"  Content: {post['content'][:100]}...")

    print("\n✓ watcher.py OK")

await test_watcher()

---
## Cell 14 — `main.py` (Monitoring Penuh)
Jalankan loop monitoring lengkap dengan FilterEngine, Database, dan Notifier.

> ⚠️ **Isi `GROUP_ID`, `KEYWORDS`, dan opsional `BOT_TOKEN`/`CHAT_ID` untuk Telegram.**  
> Loop ini akan berjalan terus. Hentikan dengan tombol **Stop** di Colab atau `Ctrl+C`.

In [ ]:
import sys, asyncio, logging, os
sys.path.insert(0, 'src')

# ── KONFIGURASI ──────────────────────────────────────────────────────────────
GROUP_IDS        = ["27422186357398808"]   # list group ID yang dipantau
KEYWORDS         = ["lelang", "jual", "auction", "bid"]
BLACKLIST        = ["spam"]
INTERVAL_SECONDS = 60                       # jeda antar scan (detik)
BOT_TOKEN        = ""                       # opsional: Telegram bot token
CHAT_ID          = ""                       # opsional: Telegram chat ID
# ─────────────────────────────────────────────────────────────────────────────

from src.logger_setup import setup_logging
from src.database import Database
from src.filter_engine import FilterEngine
from src.watcher import GroupWatcher

setup_logging(level=logging.INFO)
logger = logging.getLogger("main")

async def run_monitoring():
    session_file = os.path.join('data', 'session.json')
    if not os.path.exists(session_file):
        print("⚠  Sesi tidak ditemukan. Jalankan Cell 6 (auth.py) dulu.")
        return

    # Init database
    db = Database()
    await db.init()

    # Tambahkan keywords ke DB
    for kw in KEYWORDS:
        await db.add_keyword(kw)

    # Setup filter engine
    filter_engine = FilterEngine(keywords=KEYWORDS, blacklist=BLACKLIST)

    # Setup notifier (Telegram jika tersedia, fallback ke console)
    notifier = None
    if BOT_TOKEN and CHAT_ID:
        from src.telegram_notifier import TelegramNotifier
        notifier = TelegramNotifier(bot_token=BOT_TOKEN, chat_id=CHAT_ID)
        logger.info("Telegram notifier aktif.")
    else:
        from src.notifier import Notifier
        notifier = Notifier()
        logger.info("Console notifier aktif (Telegram belum dikonfigurasi).")

    # Jalankan watcher
    watcher = GroupWatcher(
        group_ids=GROUP_IDS,
        session_file=session_file,
        interval_seconds=INTERVAL_SECONDS,
        headless=True,
        db=db,
        filter_engine=filter_engine,
        notifier=notifier,
    )

    logger.info(f"Memulai monitoring: groups={GROUP_IDS}, keywords={KEYWORDS}")
    await watcher.run()

# Jalankan — hentikan dengan tombol Stop di Colab
try:
    await run_monitoring()
except KeyboardInterrupt:
    print("\n[INFO] Monitoring dihentikan.")